## <center> Predicting Movie Rental Durations <center>

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import operator as op

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [159]:
# Import the rental_info.csv dataset
rental_df = pd.read_csv("datasets/rental_info.csv", parse_dates=["rental_date", "return_date"])
rental_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401


In [160]:
# Review the data frame info
rental_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   rental_date       15861 non-null  datetime64[ns, UTC]
 1   return_date       15861 non-null  datetime64[ns, UTC]
 2   amount            15861 non-null  float64            
 3   release_year      15861 non-null  float64            
 4   rental_rate       15861 non-null  float64            
 5   length            15861 non-null  float64            
 6   replacement_cost  15861 non-null  float64            
 7   special_features  15861 non-null  object             
 8   NC-17             15861 non-null  int64              
 9   PG                15861 non-null  int64              
 10  PG-13             15861 non-null  int64              
 11  R                 15861 non-null  int64              
 12  amount_2          15861 non-null  float64            
 13  l

In [161]:
# Shape of the data frame
rental_df.shape

(15861, 15)

In [162]:
# Create a 'rental_length_days' columns by subtracting the rental_date from the return_date
rental_df["rental_length_days"] = rental_df["return_date"] - rental_df["rental_date"]
rental_df["rental_length_days"] = rental_df["rental_length_days"].dt.days

In [163]:
# Examine the first few lines of the rental_df data frame
rental_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length_days
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,3
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,7
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,4


In [164]:
# Examine the values of the 'special_features' column
rental_df["special_features"].unique()

array(['{Trailers,"Behind the Scenes"}', '{Trailers}',
       '{Commentaries,"Behind the Scenes"}', '{Trailers,Commentaries}',
       '{"Deleted Scenes","Behind the Scenes"}',
       '{Commentaries,"Deleted Scenes","Behind the Scenes"}',
       '{Trailers,Commentaries,"Deleted Scenes"}',
       '{"Behind the Scenes"}',
       '{Trailers,"Deleted Scenes","Behind the Scenes"}',
       '{Commentaries,"Deleted Scenes"}', '{Commentaries}',
       '{Trailers,Commentaries,"Behind the Scenes"}',
       '{Trailers,"Deleted Scenes"}', '{"Deleted Scenes"}',
       '{Trailers,Commentaries,"Deleted Scenes","Behind the Scenes"}'],
      dtype=object)

In [165]:
# Delete the '}', '{', and '"' in 'special_features'
rental_df["special_features"] = rental_df["special_features"].str.replace('{', '')
rental_df["special_features"] = rental_df["special_features"].str.replace('}', '')
rental_df["special_features"] = rental_df["special_features"].str.replace('"', '')

In [166]:
# Convert values in the 'special_features' column to lists
rental_df["special_features"] = rental_df["special_features"].to_list()

In [167]:
# Create the column 'deleted_scenes' and assign '1' to rows that contain the phrase 'Deleted Scenes'
for index, value in rental_df["special_features"].items():
    if op.contains(value, 'Deleted'):
        rental_df.loc[index, "deleted_scenes"] = 1
    else:
        rental_df.loc[index, "deleted_scenes"] = 0

In [181]:
# Sample the data frame to ensure the 'deleted_scenes' column contains correct values
rental_df.sample(n=15, replace = False)

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length_days,deleted_scenes
10010,2005-08-18 05:34:13+00:00,2005-08-24 08:38:13+00:00,5.99,2004.0,2.99,179.0,10.99,Behind the Scenes,0,0,0,0,35.8801,32041.0,8.9401,6,0.0
2398,2005-05-31 20:23:02+00:00,2005-06-04 00:00:02+00:00,4.99,2010.0,4.99,118.0,24.99,"Trailers,Behind the Scenes",0,0,1,0,24.9001,13924.0,24.9001,3,0.0
11383,2005-05-30 22:07:00+00:00,2005-06-02 18:37:00+00:00,4.99,2009.0,4.99,60.0,19.99,Commentaries,1,0,0,0,24.9001,3600.0,24.9001,2,0.0
14717,2005-08-02 23:44:46+00:00,2005-08-12 01:43:46+00:00,3.99,2005.0,0.99,136.0,25.99,"Trailers,Commentaries,Deleted Scenes",1,0,0,0,15.9201,18496.0,0.9801,9,1.0
13796,2005-07-06 21:02:33+00:00,2005-07-12 21:32:33+00:00,5.99,2007.0,4.99,150.0,19.99,"Trailers,Commentaries,Deleted Scenes",0,1,0,0,35.8801,22500.0,24.9001,6,1.0
13277,2005-07-31 00:06:24+00:00,2005-08-07 02:36:24+00:00,5.99,2009.0,2.99,147.0,24.99,"Trailers,Deleted Scenes,Behind the Scenes",0,0,1,0,35.8801,21609.0,8.9401,7,1.0
1019,2005-07-08 07:06:59+00:00,2005-07-17 08:44:59+00:00,6.99,2007.0,4.99,73.0,17.99,"Deleted Scenes,Behind the Scenes",0,1,0,0,48.8601,5329.0,24.9001,9,1.0
12231,2005-07-08 22:54:07+00:00,2005-07-17 19:09:07+00:00,8.99,2009.0,4.99,123.0,18.99,"Trailers,Commentaries,Deleted Scenes",0,0,0,0,80.8201,15129.0,24.9001,8,1.0
9407,2005-07-08 23:09:59+00:00,2005-07-15 04:51:59+00:00,8.99,2010.0,4.99,148.0,20.99,Commentaries,0,1,0,0,80.8201,21904.0,24.9001,6,0.0
7755,2005-08-22 05:45:18+00:00,2005-08-29 01:24:18+00:00,2.99,2006.0,2.99,141.0,21.99,Commentaries,0,0,1,0,8.9401,19881.0,8.9401,6,0.0
